# Evaluación de regresión lineal múltiple — Taxi IA

**Modelo:** `LinearRegression` ordinaria. **Objetivo:** tiempo estimado por arista OSM. **Evaluación:** aristas no vistas dentro del dataset operativo (no mediciones directas de viajes reales).

## 1. Preparación y resultados
Primero, desde la raíz del proyecto:

```powershell
python .\backend\scripts\ml\evaluar_regresion_red_osm.py
```

El script utiliza el modelo `.joblib` que ya entrenaste, sin modificarlo ni volver a ajustarlo. **Antes debes reentrenar una vez con la versión actual de scikit-learn**.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

raiz = Path.cwd().resolve()
for p in [raiz, *raiz.parents]:
    if (p / "backend/datos/ml").exists():
        raiz = p
        break
salida = raiz / "backend/datos/ml/modelos_regresion_osm/evaluacion_detallada"
ruta = salida / "resumen_evaluacion.json"
assert ruta.exists(), "Ejecuta primero evaluar_regresion_red_osm.py desde la raíz"
r = json.loads(ruta.read_text(encoding="utf-8"))
display(
    pd.DataFrame(
        [r["entrenamiento_final"], r["prueba"], r["referencia_prueba"]],
        index=["Entrenamiento final", "Prueba LinearRegression", "Referencia prueba"],
    ).round(6)
)

## 2. Función de pérdida (loss = MSE)

La regresión lineal ordinaria minimiza la suma de errores cuadrados; para un conjunto fijo esto equivale a minimizar el MSE:

$$MSE = \frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2$$

Como el objetivo está expresado en minutos, el MSE tiene unidades de **minutos cuadrados**. No existe una curva de *loss por épocas* en el ajuste ordinario de scikit-learn.

In [ ]:
display(Image(filename=str(salida / "04_loss_mse.png")))

## 3. Predicciones frente a valores objetivo
La línea discontinua representa predicción perfecta. La proximidad de los puntos permite evaluar el ajuste visual.

In [ ]:
display(Image(filename=str(salida / "01_objetivo_vs_predicho.png")))

## 4. Análisis de residuos
Residuo = tiempo objetivo del escenario − tiempo predicho. Distribución centrada en cero es deseable, aunque no garantiza ausencia de sesgo por segmentos.

In [ ]:
display(Image(filename=str(salida / "02_residuos.png")))

## 5. Análisis segmentado por hora y tipo de vía

In [ ]:
display(Image(filename=str(salida / "03_mae_por_hora.png")))
display(pd.read_csv(salida / "errores_por_tipo_via.csv").round(6))

## 6. Conclusiones para el informe
- La regresión lineal múltiple (`LinearRegression`) utiliza distancia, hora codificada cíclicamente, día y tipo de vía.
- El MSE es la pérdida utilizada para describir la magnitud del error cuadrático. MAE, RMSE y R² complementan la evaluación.
- Se preservó separación por aristas entre entrenamiento y prueba.
- Las predicciones se limitan a tiempos estrictamente positivos según velocidades operativas entre 5 y 80 km/h; el informe cuenta cuántas requirieron ajuste. La restricción no modifica el estimador `LinearRegression`.
- Las etiquetas `tiempo_min` proceden de velocidades estimadas/derivadas; los resultados no son evidencia de exactitud sobre viajes reales.
- Los resultados segmentados ayudan a comprobar si el error se concentra por tipo de vía u horario.

**No se utilizó red neuronal artificial ni regresión Ridge.**